In [1]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [2]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_553/FORS2.2021-08-21T06_02_18.000/Mrk-553_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_553/FORS2.2021-08-21T06_02_18.000/Mrk-553_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     361   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        42   (2094, 964)   float32   


In [3]:
image_cut = image[75:175, 0:1890]
image_error_cut = image_error[75:175, 0:1890]

In [4]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.01505 
l_HA_z = 6563 * (1+z)
l_HB_z = 4861 * (1+z) 
l_NII_z = 6583 * (1+z)
l_SII_z = 6717 * (1+z)
l_OIII_z = 5007 * (1+z)

l_HA = 6654
l_HB = 4929 
l_NII = 6676
l_SII = 6811
l_OIII = 5077

In [5]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [6]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_553/FORS2.2021-08-21T06_02_18.000/PSF/Real_seeing_Mrk553.txt")
sigma = seeing[-1] / 2.35 
x_axes = np.arange(0, 100, 1)

In [7]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, a, b):
    func = model_convolved(x, params1, params2, params3, params4)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    purple_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    sum_area = blue_area + green_area +purple_area + orange_area
    total_area = total_integral(x, params1, params2, params3, params4, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (purple_area*100)/sum_area, (orange_area*100)/sum_area]     

## Halpha

In [8]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))
fit_HA = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_553/FORS2.2021-08-21T06_02_18.000/Fit/4_peaks/Halpha_fit.csv", index_col = 0)

In [9]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 1'].iloc[3] -sigma, 
                                                                fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 2'].iloc[3] -sigma, 
                                                                fit_HA['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 3'].iloc[3] -sigma, 
                                                                fit_HA['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 4'].iloc[3] -sigma, 
                                                                fit_HA['Component 4'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

4.3667341528180135
5.4101679858518805
6.506243790291556
6.543789106314053
2.343077231719125
2.3440519836927454
1.3297034604024318
1.3529106566960147
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    83.262850  25.143669  12.472979   5.814737
Green   12.182561  62.101341   7.576148   0.004530
Purple   3.657968  11.193978  71.955314  17.465493
Orange   0.896621   1.561012   7.995559  76.715240


In [10]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 4'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

18.905475417697044
22.38935934346623
12.822325765070262
12.9709456243229
4.6856280144457605
4.687669747663763
3.2036237397644256
3.2445240656386716
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    78.494298  28.572155  12.613113   6.279376
Green   16.115893  58.403255   8.432297   0.008090
Purple   4.350437  11.436473  70.922660  20.865341
Orange   1.039372   1.588117   8.031930  72.847193


In [11]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], 
                                                                fit_HA['Component 4'], fit_HA['Component 4'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

23.90734410347323
27.53252851978337
21.10457454091051
21.641441519374723
9.466463459588507
9.471207878118243
4.276362135406533
4.318781710255184
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    73.857014  32.373845  13.070778   6.629798
Green   20.021126  52.550906  12.169523   0.011086
Purple   4.966470  13.322123  66.664252  22.659737
Orange   1.155389   1.753126   8.095447  70.699379


## HBeta

In [12]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_553/FORS2.2021-08-21T06_02_18.000/Fit/4_peaks/Hbeta_fit.csv", index_col = 0)

In [13]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -sigma, 
                                                                fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -sigma, 
                                                                fit_HA['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -sigma, 
                                                                fit_HA['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -sigma, 
                                                                fit_HA['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

1.7138325842271875
1.777936247424533
2.4132921124327398
2.407671542048335
1.4859271610993292
1.4865259570458529
0.8420576465772884
0.8455845043109242
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    67.725933  13.883687   1.199623   0.165288
Green    6.915146  38.338812   2.339111   0.002285
Purple  25.358922  47.777501  96.461266  72.631690
Orange   0.000000   0.000000   0.000000  27.200736


In [14]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

7.40757137561266
7.59826438057874
4.838902218740069
4.8270527094338185
2.9679161752054286
2.9690944723214003
2.0625859195970415
2.066190409619065
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    61.360351  17.251070   1.237405   0.184280
Green    9.275644  35.143446   2.602674   0.003762
Purple  29.364005  47.605484  96.159921  77.761527
Orange   0.000000   0.000000   0.000000  22.050430


In [15]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


9.358492302072918
9.544003391881484
8.30463342904388
8.295799993751674
5.919191190731556
5.921441723067181
2.8748377776388656
2.878712990145731
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    55.791725  20.603561   1.400952   0.189267
Green   11.635611  29.777685   3.808265   0.004756
Purple  32.572664  49.618754  94.790782  78.403472
Orange   0.000000   0.000000   0.000000  21.402504


## NII

In [16]:
mask_NII = (wv > l_NII-8)*(wv < l_NII+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_553/FORS2.2021-08-21T06_02_18.000/Fit/4_peaks/NII_fit.csv", index_col = 0)

In [17]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -sigma, 
                                                                fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -sigma, 
                                                                fit_HA['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -sigma, 
                                                                fit_HA['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -sigma, 
                                                                fit_HA['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

1.4956243540873155
1.5010372623485129
2.304946208017882
2.3262367682636538
1.839402199233037
1.8403407589414558
1.1494492362072415
1.15221758168372
              Peak 1        Peak 2     Peak 3        Peak 4
Blue    5.831376e+01  8.391505e+00   0.000007  2.845000e-24
Green   1.618711e+01  4.521819e+01  16.979864  1.155521e+01
Purple  2.549913e+01  4.639030e+01  83.018716  5.421404e+01
Orange  3.526224e-10  9.822529e-08   0.001414  3.423075e+01


In [18]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

6.702372773461782
6.714016119619895
4.610925674359026
4.645929565379633
3.6608937164392827
3.6625954047672775
2.8105285834928306
2.814938001685253
              Peak 1        Peak 2     Peak 3        Peak 4
Blue    5.246267e+01  1.145682e+01   0.000027  1.106925e-21
Green   1.899476e+01  4.215649e+01  17.189378  1.209636e+01
Purple  2.854257e+01  4.638669e+01  82.808538  5.821597e+01
Orange  1.155211e-09  1.668841e-07   0.002057  2.968767e+01


In [19]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

8.636395066375588
8.644956097267793
7.954542599209859
7.998556377209363
7.2068252717965375
7.209583654193844
3.8222556687861293
3.8276772428402333
              Peak 1        Peak 2     Peak 3        Peak 4
Blue    4.713769e+01  1.398085e+01   0.000620  2.217759e-20
Green   2.174385e+01  3.742297e+01  18.030522  1.249065e+01
Purple  3.111846e+01  4.859618e+01  81.962552  6.020981e+01
Orange  2.937280e-09  8.963812e-07   0.006305  2.729954e+01


## SII

In [20]:
mask_SII = (wv > l_SII-8)*(wv < l_SII+8)
x_axes = np.arange(0, 100, 1)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_553/FORS2.2021-08-21T06_02_18.000/Fit/4_peaks/SII_fit.csv", index_col = 0)

In [21]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -sigma, 
                                                                fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -sigma, 
                                                                fit_HA['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -sigma, 
                                                                fit_HA['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -sigma, 
                                                                fit_HA['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

1.2015660285097702
1.2071314539758018
1.9391415907042755
1.9422076258538343
1.74715583067036
1.7476429698087739
1.1522043712263987
1.1580079004833765
           Peak 1     Peak 2        Peak 3        Peak 4
Blue    45.045397   3.292440  7.623257e-09  6.481615e-40
Green   50.706658  79.026831  4.225010e+01  1.631330e+01
Purple   2.393817  14.414423  4.989994e+01  1.498265e+01
Orange   1.854128   3.266305  7.849956e+00  6.870406e+01


In [22]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

5.44560597706842
5.459437816557983
3.9070805705437657
3.9121414050125
3.490029168368192
3.490972980248515
2.8345435400384082
2.8439180853682964
           Peak 1     Peak 2        Peak 3        Peak 4
Blue    38.780535   5.297143  8.625437e-08  4.812863e-35
Green   56.150592  76.911493  4.246959e+01  1.748495e+01
Purple   3.010869  14.537013  4.962950e+01  1.865266e+01
Orange   2.058005   3.254351  7.900909e+00  6.386238e+01


In [23]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

7.089346322913933
7.100846925078307
6.8811821206278765
6.889067140123498
6.9480807548887435
6.94983609420273
3.889624142420317
3.9002101171524672
           Peak 1     Peak 2     Peak 3        Peak 4
Blue    33.675223   7.362373   0.000011  1.143083e-32
Green   60.433826  72.420325  43.341415  1.798804e+01
Purple   3.663098  16.815341  48.551997  1.994109e+01
Orange   2.227853   3.401961   8.106577  6.207087e+01


## OIII

In [24]:
mask_OIII = (wv > l_OIII-8)*(wv < l_OIII+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_553/FORS2.2021-08-21T06_02_18.000/Fit/4_peaks/OIII_fit.csv", index_col = 0)

In [25]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -sigma, 
                                                                fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -sigma, 
                                                                fit_HA['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -sigma, 
                                                                fit_HA['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -sigma, 
                                                                fit_HA['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

2.4726146258524637
2.3311630683681237
2.8136642625703208
2.7380933511749572
1.9579855713592522
1.958451959585335
1.1560787889580637
1.1680324452683033
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    71.243020  15.719890   0.229126   0.002471
Green   19.782061  57.837820  33.167453  22.763027
Purple   8.751249  25.871996  64.675421  41.023488
Orange   0.223670   0.570295   1.928000  36.211015


In [26]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -2*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

10.498157669997024
9.967712327156018
5.7454062108613915
5.593707281095373
3.907200458575433
3.9081615808056935
2.8782556773204857
2.9027670652384834
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    65.407840  20.161144   0.249815   0.003113
Green   23.923339  53.907447  33.412976  23.503755
Purple  10.403574  25.370309  64.390838  44.153023
Orange   0.265247   0.561100   1.946371  32.340108


In [27]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 1'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 2'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 3'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit_HA['Component 4'].iloc[3] -3*sigma, 
                                                                fit_HA['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

13.028518732754547
12.434092154305096
10.104331381793122
9.837191617842073
7.7573004512839905
7.7593386363377235
4.007123247429198
4.041652301281441
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    60.134584  24.198174   0.348913   0.003420
Green   27.709506  48.667184  34.388393  23.717606
Purple  11.854400  26.544140  63.243766  44.748760
Orange   0.301509   0.590502   2.018929  31.530215
